# FLUX.2 [flex] Demo

**Black Forest Labs x Microsoft**

This notebook demonstrates FLUX.2 [flex] capabilities — our model optimized for text-heavy design workflows, UI prototyping, and creative production.

## What Makes [flex] Different

FLUX.2 [flex] excels where precise typography matters:

- **Text Rendering** — Clean, readable text for logos, UI copy, product packaging, and social graphics
- **Detail Preservation** — Maintains sharpness in brand assets across any resolution
- **Fine-Grained Control** — Adjust inference steps and guidance scale for production-quality outputs
- **Multi-Reference Support** — Up to 8 reference images via API for complex compositing

## What We'll Cover

1. **Text-to-Image Generation** — Posters, packaging, and graphic design with accurate typography
2. **Hex Color Matching** — Precise brand color specification
3. **Product Photography** — Clean studio shots with text and branding
4. **Image Editing** — Character consistency

![Realism](https://cdn.sanity.io/images/2gpum2i6/production/41055678178f6fe75ca618b854b195e48dfc55ed-2127x1400.jpg)

---

## Prerequisites

- **For API demos**: BFL API key from [dashboard.bfl.ai](https://dashboard.bfl.ai/get-started)
- Also available on [Azure AI Foundry](https://ai.azure.com/catalog/models/FLUX.2-flex)

---

## Setup

In [ ]:
# Install dependencies (uncomment if needed)
# !pip install requests matplotlib pillow torch diffusers transformers accelerate peft bitsandbytes fal-client 

import os
import time
import base64
from io import BytesIO
from pathlib import Path
from typing import Optional, List

import requests
from PIL import Image
from IPython.display import display, HTML

from dotenv import load_dotenv

load_dotenv()

print("Setup complete!")

In [ ]:
# Configuration
BFL_API_KEY = os.environ.get("BFL_API_KEY")

API_BASE = "https://api.bfl.ai/v1"

OUTPUT_DIR = Path("./outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

def get_headers():
    """Get headers for BFL API requests."""
    return {
        "accept": "application/json",
        "x-key": BFL_API_KEY,
        "Content-Type": "application/json",
    }

---

# Part 1: Text-to-Image with FLUX.2 [flex]

FLUX.2 [flex] is designed for design workflows where typography and fine details matter. It handles multi-word headlines, body copy, and editorial layouts with correctly spaced, readable fonts.

### Use Cases

| Category | Examples |
|----------|----------|
| **Brand & Graphic Design** | Logos, packaging, marketing materials with clean typography |
| **Creative Production** | Headlines, body copy blocks, editorial layouts |
| **UI Prototyping** | Dashboards, button labels, navigation copy |
| **Product Design** | Mockups with accurate on-screen text |

## 1.1 Helper Functions

In [ ]:
def submit_request(endpoint: str, payload: dict) -> dict:
    """Submit a request to the BFL API."""
    url = f"{API_BASE}/{endpoint}"
    response = requests.post(url, headers=get_headers(), json=payload)
    response.raise_for_status()
    return response.json()


def poll_result(polling_url: str, max_wait: int = 120) -> dict:
    """Poll for the result of an async request."""
    start = time.time()
    
    while time.time() - start < max_wait:
        response = requests.get(polling_url, headers={"accept": "application/json", "x-key": BFL_API_KEY})
        result = response.json()
        
        status = result.get("status")
        if status == "Ready":
            return result["result"]
        elif status == "Failed":
            raise Exception(f"Generation failed: {result.get('error', 'Unknown error')}")
        
        time.sleep(0.5)
    
    raise TimeoutError(f"Request timed out after {max_wait}s")


def generate_image(endpoint: str, payload: dict, show: bool = True, max_width: int = 600) -> Image.Image:
    """Generate an image and optionally display it."""
    print(f"Submitting request to {endpoint}...")
    response = submit_request(endpoint, payload)
    
    request_id = response["id"]
    polling_url = response["polling_url"]
    cost = response.get("cost", "N/A")
    
    print(f"Request ID: {request_id}")
    print(f"Cost: {cost} credits")
    print("Waiting for result...")
    
    result = poll_result(polling_url)
    
    image_url = result["sample"]
    image_response = requests.get(image_url)
    image = Image.open(BytesIO(image_response.content))
    
    if show:
        # Display at native quality with controlled size
        buffer = BytesIO()
        image.save(buffer, format='PNG')
        img_data = base64.b64encode(buffer.getvalue()).decode()
        display(HTML(f'<img src="data:image/png;base64,{img_data}" style="max-width: {max_width}px; height: auto; border-radius: 8px;">'))
    
    return image

## 1.2 Text-to-Image Generation

Let's see [flex] in action with design prompts that require accurate typography. 

In [ ]:
# Coconut Poster 

prompt = """
Bold modern poster design, large halved coconut dominating the center-right of the composition, clean flat illustration style with subtle paper-like texture, 
the coconut shell a rich dark brown and the inner white flesh smooth and bright, slight soft shadow beneath to give gentle depth. 
Background is a solid sky blue color, minimal and uncluttered. At the top left, big heavy bold sans-serif lettering in dark brown reading "COCO" on the first line and "NUT." 
on the second line, tight leading and strong presence. At the bottom right corner, small clean white sans-serif text reading "TROPICAL HARVEST" aligned right. 
Overall look is minimalistic, flat vector-style graphic with balanced negative space and strong contrast, suitable for a contemporary product or event poster.
""".strip()

payload = {
    "prompt": prompt,
    "width": 2048,
    "height": 1728,
    "seed": 328583
}

image = generate_image("flux-2-flex", payload)

In [ ]:
# More examples showcasing typography and graphic design

# Graphic Design poster - note how [flex] handles stacked text and multiple text elements
record_poster = """
Graphic design poster, minimalist flat illustration on a deep magenta background, featuring an oversized black vinyl record partially sliding out of a 
simple white sleeve positioned slightly off-center, clean geometric shapes and crisp edges, no shading or gradients,
flat black and white illustration with minimal detail, modern bold white sans-serif typography at the top reading "VINYL.", 
in the lower right corner small yellow sans-serif text labels reading "ANALOG SOUND" and "33 RPM", 
high contrast color palette, strong negative space, contemporary music-themed wall art poster, 
centered composition with generous margins, sharp vector-style look
""".strip()

# Sneakers design - multiple text elements at different sizes and positions
sneakers_poster = """
Retro graphic poster design, minimalist vector style, featuring a large pair of white high-top sneakers as the central subject,
drawn in clean bold outlines and flat colors without any gradients, 
sneakers angled slightly to the left with visible laces and rubber soles, placed prominently on a solid cobalt blue background,
subtle paper grain texture over the entire image to evoke a printed vintage poster feel,
bold playful stacked pink text on the right side of the composition reading "SNEAKERS" in large letters,
aligned vertically and taking up most of the right third of the poster, at the bottom left in small uppercase white text the phrases "STREET CULTURE" and "SINCE 1987" 
arranged in two lines, balanced negative space, sharp edges, high contrast, modern yet nostalgic streetwear aesthetic.
""".strip()

print("Generating vinyl record poster...")
arch_image = generate_image("flux-2-flex", {"prompt": record_poster, "width": 2048, "height": 1728, "seed": 916153}, show=True)

print("\nGenerating sneakers poster...")
product_image = generate_image("flux-2-flex", {"prompt": sneakers_poster, "width": 2048, "height": 1728, "seed": 303845}, show=True)

## 1.3 Hex Color Matching

When you're working with brand guidelines, "blue" isn't specific enough. [flex] understands hex codes directly in prompts, so you can match exact brand colors without post-processing or color correction. Useful for agencies and in-house teams who need assets that fit existing style guides.

In [ ]:
# Brand colors via hex codes
brand_prompt = """
A modern living room interior. The accent wall is painted in color #003049.
A plush velvet sofa in color #D62828 sits against it. 
Throw pillows in color #F77F00 and color #FCBF49 add contrast. 
Natural daylight creates a warm, inviting atmosphere. 
Interior design magazine style photography.
""".strip()

payload = {
    "prompt": brand_prompt,
    "width": 2048,
    "height": 1728,
    "seed": 123
}

brand_image = generate_image("flux-2-flex", payload, show=True)

## 1.4 Product Photography

Traditional product photography requires studio time, lighting setups, and multiple rounds of retouching. With [flex], you can generate consistent product shots with accurate text and branding — useful for early-stage mockups, A/B testing packaging concepts, or creating variations at scale before committing to a physical shoot.

In [ ]:
generate_image(
    "flux-2-flex",
    {
        "prompt": 'Product photography style, ultra-clean studio shot of three ceramic plant pots arranged in a neat row on a pure white tabletop, photographed from a slightly elevated three-quarter front angle with soft diffused natural window light coming from the left, creating gentle highlights on the left sides and very soft shadows falling to the right. The background is seamless white with no distractions. The left pot is a matte celtic blue (#3971b8), the center pot is a matte tea green (#c8d69b), and the right pot is a matte vanilla (#f6e6a5). Each pot has a smooth cylindrical shape with a slightly tapered base, consistent size and proportions, and crisp, clean rims. On the front of each pot is a cute kawaii plant mascot character printed in a high-contrast color suited to each pot, showing a small smiling potted plant with minimal dot eyes, a tiny curved mouth, two short legs, and three simple rounded leaves on top. Below each mascot, centered on the pot, the text "Leafy Friends" is printed in a rounded sans-serif font in the style of Comfortaa, with the subline "Plant Care" in a smaller size directly beneath, both perfectly aligned and sharply printed. Subtle, soft-edged shadows appear under each pot on the white surface to emphasize depth, with a shallow depth of field that keeps all three pots in sharp focus while the background remains smoothly blurred and bright.',
        "width": 1440,
        "height": 1056,
        "seed": 177430},
    show=False
)

## Product Photography with an Input Image

Got a quick phone photo of a product? [flex] can transform it into a clean, editorial-style shot — removing distractions, adjusting the background, and enhancing the product while preserving its details.

### Input
![image](https://cdn.bfl.ai/files/bb3f8bf7-cf8e-4f54-b1f3-efa1483d3431/generations/af72fa09-1cf0-483a-8327-dfd2330646a8/raw?sv=2025-07-05&se=2026-03-09T15%3A16%3A05Z&sr=b&sp=r&sig=dcVW1q5awZg6xFtty5S77YPB6%2FtdA2iHf3MTwRJHl5o%3D)

In [ ]:
generate_image(
    "flux-2-flex",
    {
        "prompt": 'Transform this scene into a magazine-style editorial product photograph focused solely on the orange lamp. Crop in tightly so the orange lamp becomes the central subject, keeping its current perspective and lighting. Remove the small fan, wall outlet, bed, and other distractions, replacing them with a clean, softly blurred neutral background that complements the lamp’s color. Preserve the lamp’s exact shape, color, and surface details while enhancing contrast and clarity slightly.',
        "input_image":"https://cdn.bfl.ai/files/bb3f8bf7-cf8e-4f54-b1f3-efa1483d3431/generations/af72fa09-1cf0-483a-8327-dfd2330646a8/raw?sv=2025-07-05&se=2026-03-09T15%3A16%3A05Z&sr=b&sp=r&sig=dcVW1q5awZg6xFtty5S77YPB6%2FtdA2iHf3MTwRJHl5o%3D",
        "width": 960,
        "height": 1440,
        "seed": 784648},
    show=False
)

In [ ]:
generate_image(
    "flux-2-flex",
    {
        "prompt": 'Photorealistic storefront window sign for a plant shop, composed and lit like a professional product and interior design photograph, viewed straight-on through a clean glass window with soft reflections of faint ambient outdoor light. In the center of the scene, a square wooden frame sign hangs or stands at eye level, its wood a natural light oak tone with subtle grain visible, surrounding a thick ivory (#fbfcee) matte paper insert. On the paper, centered near the top, is a simple dark green (#343b1b) plant icon: three teardrop-shaped leaves fanning upward symmetrically from a single short stem, minimal and flat with no outline. Below the icon, the shop name is printed in two stacked lines: first line "PLANT" and directly underneath second line "SHOP", both rendered in a playful rounded sans-serif typeface in the style of Comfortaa, with a casual mix of uppercase and lowercase letter heights and sizes, the letters colored in celtic blue (#3971b8). Ensure the text "FLUX PLANT SHOP" is clean, legible, and centered under the icon, with balanced spacing. Behind the sign, softly out of focus through the glass, multiple lush green potted plants of various leaf shapes fill warm wooden shelves, in terracotta and white ceramic containers, with warm interior lighting casting a cozy golden glow. The depth of field is shallow, with the sign sharply in focus, background plants gently blurred, and subtle window reflections giving a real-world street storefront feel.',
        "width": 1440,
        "height": 1056,
        "seed": 483100},
    show=False
)

---

# Part 2: Image Editing with FLUX.2 [flex]

Beyond text-to-image, [flex] handles image-to-image transformations — background swaps, style transfers, and character consistency across scenes.

### Key Principle
Reference images carry the visual details. Your prompt describes *what should change*, not what the image looks like.

### Edit Types
| Type | Description | Example Prompt |
|------|-------------|----------------|
| **Background swap** | Change the environment | "Place her in a coffee shop" |
| **Style transfer** | Apply a new visual style | "Turn into a watercolor painting" |
| **Object replacement** | Swap specific elements | "Replace the bike with a horse" |
| **Element addition** | Add new objects | "Add a cat sleeping on the chair" |
| **Attribute change** | Modify properties | "Change the dress from blue to red" |

In [ ]:
# Helper for image editing
def image_to_base64(image: Image.Image) -> str:
    """Convert a PIL Image to base64 string."""
    buffer = BytesIO()
    image.save(buffer, format="PNG")
    return base64.b64encode(buffer.getvalue()).decode("utf-8")

## Examples

### input
![Input Image](https://cdn.bfl.ai/files/9d250a50-5a6d-4a1a-b7a7-c2fa0fe04422/generations/aa60f4ef-0371-4d3d-a3d2-7e0fef2616f1/raw?sv=2025-07-05&se=2026-03-08T10:19:09Z&sr=b&sp=r&sig=HqA7H2aEK9ot77G/DYFCU1caEUx7XUscndAQJc/OGcs%3D)

In [ ]:
generate_image(
    "flux-2-flex",
    {
        "prompt": "Extract the complete front-facing packaging design from the can, including all artwork, colors, typography, and layout, and transform it into a perfectly flat 2D label centered on a clean, solid white background. Remove the cylindrical can form, metallic top, ice cubes, reflections, depth of field, and surrounding environment. Preserve original colors, graphics, and text exactly as shown, with sharp, high-contrast edges and no added shadows or gradients on the background.",
        "input_image": "https://cdn.bfl.ai/files/9d250a50-5a6d-4a1a-b7a7-c2fa0fe04422/generations/aa60f4ef-0371-4d3d-a3d2-7e0fef2616f1/raw?sv=2025-07-05&se=2026-03-08T10:19:09Z&sr=b&sp=r&sig=HqA7H2aEK9ot77G/DYFCU1caEUx7XUscndAQJc/OGcs%3D",
        "seed": 547972},
    show=False
)

## Extending a Design into a Scene

Start with a 2D graphic or poster, then use [flex] to imagine how it would look in a real environment — a storefront, an interior, a billboard. Useful for pitching concepts to clients or visualizing brand applications.

In [ ]:
generate_image(
    "flux-2-flex",
    {
        "prompt": 'Transform this image into a wide interior view of a coffee shop that keeps the original poster artwork unchanged on one wall, and extends its aesthetic into the space: lime-green walls matching the poster background, large pink leopard graphics as wall murals, checkerboard accents on floor and counter fronts. Add bold cream pop-art typography signage reading "KOMBUCHA & COFFEE BAR" and matching menu boards. Preserve the flat, graphic, pop-art style and playful, edgy mood only.',
        "input_image": "https://cdn.bfl.ai/files/9d250a50-5a6d-4a1a-b7a7-c2fa0fe04422/generations/cd231d7b-db47-4314-a062-b4491765f153/raw?sv=2025-07-05&se=2026-03-08T10%3A20%3A19Z&sr=b&sp=r&sig=QwsHSF5%2Bjsbkj%2FeLkncp4BvUqjNaHCjWITnfJUDlDig%3D",
        "width": 1440,
        "height": 1056,
        "seed": 563213},
    show=False
)

## Character Consistency

One of the trickier problems in image generation: keeping a character looking the same across different scenes. With [flex], you provide a reference image and describe the new context — the model preserves identity while adapting pose, lighting, and environment.

## Examples

### Input Image
![Input Image](https://cdn.bfl.ai/files/9d250a50-5a6d-4a1a-b7a7-c2fa0fe04422/generations/0466ae92-e5a2-43d6-9555-57f239199a0c/raw?sv=2025-07-05&se=2026-03-08T10%3A00%3A51Z&sr=b&sp=r&sig=e3AUhC85XVkTlrGaqxNitpO3WZ9wMsbSalbBSy2nNGQ%3D)

In [ ]:
generate_image(
    "flux-2-flex",
    {
        "prompt": 'sitting on a log beside a campfire at dusk, orange firelight illuminating the underside of his face and beard, dark pine trees silhouetted against a deep blue twilight sky, sparks drifting upward.',
        "input_image": "https://cdn.bfl.ai/files/9d250a50-5a6d-4a1a-b7a7-c2fa0fe04422/generations/0466ae92-e5a2-43d6-9555-57f239199a0c/raw?sv=2025-07-05&se=2026-03-08T10%3A00%3A51Z&sr=b&sp=r&sig=e3AUhC85XVkTlrGaqxNitpO3WZ9wMsbSalbBSy2nNGQ%3D",
        "width": 1440,
        "height": 960,
        "seed": 446341},
    show=False
)

In [ ]:
generate_image(
    "flux-2-flex",
    {
        "prompt": 'the man , standing at a woodworking bench in a cluttered workshop, holding a hand plane, sawdust particles visible in a shaft of natural window light from the left, tools hanging on a pegboard wall behind him.',
        "input_image": "https://cdn.bfl.ai/files/9d250a50-5a6d-4a1a-b7a7-c2fa0fe04422/generations/0466ae92-e5a2-43d6-9555-57f239199a0c/raw?sv=2025-07-05&se=2026-03-08T10%3A00%3A51Z&sr=b&sp=r&sig=e3AUhC85XVkTlrGaqxNitpO3WZ9wMsbSalbBSy2nNGQ%3D",
        "width": 960,
        "height": 1440,
        "seed": 998119},
    show=False
)

# Get Started

## [bfl.ai](https://bfl.ai)
Sign up and start generating

## [docs.bfl.ai](https://docs.bfl.ai)
API reference and examples

## [FLUX.2 [flex] on Azure AI Foundry](https://ai.azure.com/catalog/models/FLUX.2-flex)
Deploy on Azure

---

**Black Forest Labs x Microsoft**